# 07 — Selection Operators (Tournament and Roulette)

**Maps to:** `report/Chapters/Task2.tex` §`T2:Selection`.  
**Ticket:** TICKET-07.

Tournament (vary $k$) and roulette wheel. Show selection pressure on a toy fitness vector.

In [1]:
import numpy as np
from pathlib import Path
import tsplib95

In [2]:
def load_tsp(path):
    problem     = tsplib95.load(str(path))
    nodes       = list(problem.get_nodes())
    coords      = np.array(
        [problem.node_coords[node] for node in nodes],
        dtype=np.float64
    )
    diff        = coords[:, np.newaxis, :] - coords[np.newaxis, :, :]
    dist_matrix = np.sqrt((diff**2).sum(axis=-1))
    return coords, dist_matrix


def random_population(pop_size, n_cities, rng):
    return np.array([rng.permutation(n_cities) for _ in range(pop_size)])

**Implementation**

Two selection operators are implemented for the Genetic Algorithm.

`tournament_select(population, fitnesses, k, rng)` randomly samples k chromosomes from the population and returns the one with the lowest tour length. A higher k increases selection pressure, favouring fitter individuals more strongly.

`roulette_select(population, fitnesses, rng)` assigns each chromosome a selection probability inversely proportional to its tour length. Shorter tours receive higher probabilities, giving fitter individuals a greater chance of being selected while still allowing weaker individuals to contribute.

A strategy wrapper function `select(method, ...)` allows both methods to be swapped via a single configuration parameter, making the selection operator configurable without changing the GA loop.

In [3]:
def tournament_select(population, fitnesses, k, rng):
    candidates = rng.choice(len(population), size=k, replace=False)
    winner     = candidates[np.argmin(fitnesses[candidates])]
    return population[winner].copy()


def roulette_select(population, fitnesses, rng):
    inv   = 1.0 / fitnesses
    probs = inv / inv.sum()
    idx   = rng.choice(len(population), p=probs)
    return population[idx].copy()


def select(method, population, fitnesses, rng, k=3):
    if method == 'tournament':
        return tournament_select(population, fitnesses, k, rng)
    elif method == 'roulette':
        return roulette_select(population, fitnesses, rng)
    else:
        raise ValueError(f'Unknown selection method: {method}')

**Unit Test**

Both selection operators are verified using a synthetic fitness vector of five chromosomes with known fitness values. The test confirms that each operator returns a valid tour of the correct length with no duplicate cities.

In [4]:
rng        = np.random.default_rng(42)
population = random_population(5, 4, rng)
fitnesses  = np.array([100.0, 200.0, 50.0, 300.0, 150.0])

print(f"Population:")
for i, tour in enumerate(population):
    print(f"  Tour {i}: {tour}  fitness: {fitnesses[i]}")

try:
    winner = tournament_select(population, fitnesses, k=3, rng=rng)
    assert len(winner) == 4, f"Expected length 4, got {len(winner)}"
    assert len(set(winner)) == 4, f"Duplicate cities: {winner}"
    print(f"\nTournament winner : {winner}")
    print("tournament_select test passed.")
except AssertionError as e:
    print(f"tournament_select test failed: {e}")

try:
    winner = roulette_select(population, fitnesses, rng=rng)
    assert len(winner) == 4, f"Expected length 4, got {len(winner)}"
    assert len(set(winner)) == 4, f"Duplicate cities: {winner}"
    print(f"\nRoulette winner   : {winner}")
    print("roulette_select test passed.")
except AssertionError as e:
    print(f"roulette_select test failed: {e}")

try:
    t = select('tournament', population, fitnesses, rng, k=3)
    r = select('roulette',   population, fitnesses, rng)
    assert len(set(t)) == 4
    assert len(set(r)) == 4
    print(f"\nStrategy pattern (tournament): {t}")
    print(f"Strategy pattern (roulette)  : {r}")
    print("strategy pattern test passed.")
except Exception as e:
    print(f"strategy pattern test failed: {e}")

Population:
  Tour 0: [3 2 1 0]  fitness: 100.0
  Tour 1: [3 2 0 1]  fitness: 200.0
  Tour 2: [0 3 2 1]  fitness: 50.0
  Tour 3: [2 3 0 1]  fitness: 300.0
  Tour 4: [2 3 0 1]  fitness: 150.0

Tournament winner : [3 2 1 0]
tournament_select test passed.

Roulette winner   : [2 3 0 1]
roulette_select test passed.

Strategy pattern (tournament): [2 3 0 1]
Strategy pattern (roulette)  : [0 3 2 1]
strategy pattern test passed.


**Selection Pressure Demo**

To demonstrate the difference in selection pressure between the two methods, each operator is applied 1000 times on the same synthetic population. The frequency with which each chromosome is selected reflects how strongly each method favours lower fitness values.

In [9]:
coords, dist_matrix = load_tsp(Path('../data/TSP-dataset/kroA100.tsp'))

rng        = np.random.default_rng(42)
population = random_population(5, len(coords), rng)
fitnesses  = np.array([
    sum(dist_matrix[tour[i], tour[(i+1) % len(tour)]]
        for i in range(len(tour)))
    for tour in population
])

print("Population fitnesses (tour lengths):")
for i, f in enumerate(fitnesses):
    print(f"  Tour {i}: {f:.2f}")

n_trials          = 1000
tournament_counts = np.zeros(len(population))
roulette_counts   = np.zeros(len(population))

rng = np.random.default_rng(42)
for _ in range(n_trials):
    t_winner = tournament_select(population, fitnesses, k=3, rng=rng)
    r_winner = roulette_select(population, fitnesses, rng=rng)

    for i, ind in enumerate(population):
        if np.array_equal(t_winner, ind):
            tournament_counts[i] += 1
        if np.array_equal(r_winner, ind):
            roulette_counts[i] += 1

print(f"\nSelection frequency over {n_trials} trials:")
print(f"{'Tour':>6} {'Fitness':>12} {'Tournament':>12} {'Roulette':>10}")
print("-" * 45)
for i in range(len(population)):
    print(f"  {i:>4}   {fitnesses[i]:>10.2f}   "
          f"{tournament_counts[i]:>8.0f}   {roulette_counts[i]:>8.0f}")

Population fitnesses (tour lengths):
  Tour 0: 166578.15
  Tour 1: 166765.28
  Tour 2: 174384.37
  Tour 3: 157074.39
  Tour 4: 177877.78

Selection frequency over 1000 trials:
  Tour      Fitness   Tournament   Roulette
---------------------------------------------
     0    166578.15        300        213
     1    166765.28         98        203
     2    174384.37          0        190
     3    157074.39        602        214
     4    177877.78          0        180


**END**